# 1. Go2 の歩行の学習（Genesis × PPO）

参考: https://tototototototo7.hatenablog.com/entry/2025/genesis_test_google_colab

Genesis の公式サンプル（`examples/locomotion/go2_train.py`）で、Go2 に前へ歩くことを学習させる。

- 学習・評価・動画作成まですべて Google Colab 上で行い、ローカルには何もインストールしない
- 事前に VS Code の Colab 拡張で **GPU（T4 など）ランタイム** に接続しておく
- ランタイムが切れると環境は消えるので、毎回上から実行し直す
- このノートブックだけで完結する（ほかの 2 つのノートブックを先に実行する必要はない）

In [ ]:
# GPU が割り当てられているか確認
!nvidia-smi

## 1. インストール

「セッションを再起動」を求められたら、カーネルを再起動してから次のセルに進む。

In [ ]:
!pip install -q genesis-world

In [ ]:
import importlib.metadata as md
import os
import re

GS_VER = md.version('genesis-world')
print('genesis-world', GS_VER)

# インストールした Genesis と同じバージョンのサンプルコードを取得（タグがなければ最新版）
GENESIS_DIR = '/content/Genesis'
if not os.path.exists(GENESIS_DIR):
    repo = 'https://github.com/Genesis-Embodied-AI/Genesis.git'
    if os.system(f'git clone -q --depth 1 --branch v{GS_VER} {repo} {GENESIS_DIR}') != 0:
        os.system(f'git clone -q --depth 1 {repo} {GENESIS_DIR}')
LOCO_DIR = f'{GENESIS_DIR}/examples/locomotion'

# go2_train.py が要求する rsl_rl のバージョン（例: >=5.0.0）を読み取ってインストール
m = re.search(r"rsl-rl-lib\s*([<>=!~]=?\s*[0-9.]+)", open(f'{LOCO_DIR}/go2_train.py').read())
RSL_REQ = f'rsl-rl-lib{m.group(1)}' if m else 'rsl-rl-lib'
print('install', RSL_REQ)
!pip install -q "{RSL_REQ}" tensordict tensorboard

In [ ]:
# Colab では EGL が NVIDIA GPU を見つけるための設定ファイルがなく、描画の初期化で Segmentation fault になる
# （MuJoCo の公式 Colab チュートリアルと同じ対処）
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
    with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}\n')
os.environ['PYOPENGL_PLATFORM'] = 'egl'  # 学習スクリプト（子プロセス）にも引き継がれる
!ldconfig -p | grep -i egl_nvidia

## 2. Google Drive（任意）

学習結果を残したい場合のみ `USE_DRIVE = True` にする。

In [ ]:
USE_DRIVE = True
SAVE_DIR = None

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/pai_finalwork'
    os.makedirs(SAVE_DIR, exist_ok=True)

## 3. 学習（PPO）

結果は `examples/locomotion/logs/{EXP_NAME}` に保存される。

In [ ]:
EXP_NAME = 'go2-walking'
NUM_ENVS = 4096       # 並列環境数（GPU メモリが足りなければ減らす）
MAX_ITER = 300        # 学習イテレーション数

%cd {LOCO_DIR}
# -X faulthandler: 異常終了（Segmentation fault）したときに、どこで落ちたかを表示する
!python -X faulthandler go2_train.py -e {EXP_NAME} -b {NUM_ENVS} --max-iterations {MAX_ITER}
!ls {LOCO_DIR}/logs/{EXP_NAME}

In [ ]:
# 学習結果を Drive にコピー
import shutil

if SAVE_DIR:
    shutil.copytree(f'{LOCO_DIR}/logs/{EXP_NAME}', f'{SAVE_DIR}/logs/{EXP_NAME}', dirs_exist_ok=True)
    print('saved to', f'{SAVE_DIR}/logs/{EXP_NAME}')

## 4. 評価（動画で確認）

Colab には画面がないのでビューアは使わず、カメラでオフスクリーン描画して mp4 に保存する。
`gs.init()` はカーネルごとに 1 回しか呼べないため、再実行するときはカーネルを再起動する。

In [ ]:
os.environ['PYOPENGL_PLATFORM'] = 'egl'  # genesis を import する前に設定する

import glob
import pickle
import sys

import torch
import genesis as gs
from rsl_rl.runners import OnPolicyRunner

sys.path.insert(0, LOCO_DIR)
from go2_env import Go2Env

log_dir = f'{LOCO_DIR}/logs/{EXP_NAME}'
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f'{log_dir}/cfgs.pkl', 'rb'))
reward_cfg['reward_scales'] = {}

# gs.init() と環境の作成はカーネルごとに 1 回だけ。2 回目以降はモデルの読み込みだけやり直す
if 'env' not in globals():
    gs.init(backend=gs.gpu, logging_level='warning')

    # Go2Env は内部でシーンを build してしまうので、build の直前にカメラを追加する
    cam = None
    _orig_build = gs.Scene.build

    def _build_with_camera(self, *args, **kwargs):
        global cam
        cam = self.add_camera(res=(640, 480), pos=(2.0, -2.0, 1.5), lookat=(0.0, 0.0, 0.3), fov=40, GUI=False)
        return _orig_build(self, *args, **kwargs)

    gs.Scene.build = _build_with_camera
    env = Go2Env(
        num_envs=1,
        env_cfg=env_cfg,
        obs_cfg=obs_cfg,
        reward_cfg=reward_cfg,
        command_cfg=command_cfg,
        show_viewer=False,
    )
    gs.Scene.build = _orig_build

# 一番新しいチェックポイントを読み込む
ckpt_path = max(glob.glob(f'{log_dir}/model_*.pt'), key=lambda p: int(re.findall(r'model_([0-9]+)', p)[0]))
print('load', ckpt_path)
runner = OnPolicyRunner(env, train_cfg, log_dir, device=gs.device)
runner.load(ckpt_path)
policy = runner.get_inference_policy(device=gs.device)

In [ ]:
# 録画（このセルはカーネルを再起動せずに何度でも実行できる）
NUM_STEPS = 250  # 0.02 秒 x 250 = 5 秒
VIDEO_PATH = '/content/go2_eval.mp4'

obs = env.reset()
# Genesis 1.4 以降は録画中、シミュレーションを進めるたびに自動でフレームが記録される（cam.render() は不要）
cam.start_recording(save_to_filename=VIDEO_PATH, fps=int(1 / env.dt))
with torch.no_grad():
    for _ in range(NUM_STEPS):
        # ロボットを追いかけるようにカメラを動かす
        x, y, _z = env.base_pos[0].cpu().numpy()
        cam.set_pose(pos=(x + 2.0, y - 2.0, 1.5), lookat=(x, y, 0.3))
        actions = policy(obs)
        obs, rews, dones, infos = env.step(actions)
cam.stop_recording()
print('saved', VIDEO_PATH)

In [ ]:
# 動画を表示（出力は ipynb に埋め込まれるので、見終わったら出力をクリアするとファイルが軽くなる）
from IPython.display import Video

if SAVE_DIR:
    shutil.copy(VIDEO_PATH, SAVE_DIR)
Video(VIDEO_PATH, embed=True, width=640)